geoprompt 0.2.0 ready

Network routing, service area analysis, and GeoPromptFrame spatial operations.


In [5]:
from __future__ import annotations

import json, os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import geoprompt as gp

_ROOT = Path.cwd()
if (_ROOT / "examples" / "notebooks").exists():
    OUTPUT_DIR = _ROOT / "examples" / "notebooks" / "geoprompt" / "outputs"
else:
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "0") == "1"


def fetch_json(url, fallback):
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})
        with urlopen(req, timeout=10) as r:
            return json.loads(r.read().decode("utf-8"))
    except (URLError, TimeoutError, ValueError):
        return fallback


def make_point(x: float, y: float):
    return {"type": "Point", "coordinates": [float(x), float(y)]}


def frame_from_xy(data: dict[str, list], xs: list[float], ys: list[float], crs: str = "EPSG:4326"):
    columns = list(data.keys())
    rows = []
    for i, (x, y) in enumerate(zip(xs, ys)):
        row = {col: data[col][i] for col in columns}
        row["geometry"] = make_point(x, y)
        rows.append(row)
    return gp.GeoPromptFrame.from_records(rows, crs=crs)


def frame_preview(frame: gp.GeoPromptFrame, columns: list[str], limit: int | None = None):
    rows = frame.to_records()
    if limit is not None:
        rows = rows[:limit]
    return pd.DataFrame([{col: row.get(col) for col in columns} for row in rows], columns=columns)


print(f"geoprompt {gp.__version__} ready")

geoprompt 0.2.0 ready


## Section A: Pull Data Sources


In [6]:
eia   = fetch_json("https://api.eia.gov/v2/electricity/facility-fuel/data/", {"response": {"data": []}})
openei = fetch_json("https://openei.org/services/rest/search?api_key=DEMO&q=utility", {"result": []})
print("EIA records:", len(eia.get("response", {}).get("data", [])))
print("OpenEI results:", len(openei.get("result", [])))


EIA records: 0
OpenEI results: 0


## Section B: Spatial Analysis


In [7]:
# Utility substation data
nodes_data = {
    "node":   ["PLANT", "SUB1", "SUB2", "SUB3", "SUB4", "SUB5", "SUB6"],
    "cost":   [0.0,     3.0,    8.0,    11.0,   7.0,    4.0,    13.0],
    "cap_mw": [500,     120,    90,      80,    110,    100,     70],
    "type":   ["plant", "sub",  "sub",   "sub",  "sub",  "sub",  "sub"],
}

lons = [-87.70, -87.63, -87.72, -87.65, -87.58, -87.76, -87.60]
lats = [ 41.94,  41.88,  41.83,  41.79,  41.85,  41.91,  41.75]


# 1. Build GeoPromptFrame
gdf = frame_from_xy(nodes_data, lons, lats, crs="EPSG:4326")

print("GeoPromptFrame:")
print(frame_preview(gdf, ["node", "cost", "cap_mw"]).to_string(index=False))


# 2. Buffer: service zones around substations
gdf_projected = gdf.to_crs("EPSG:3857")
gdf_projected["buffer_geom"] = gdf_projected.buffer(5000)["geometry"]  # 5 km buffer
print(f"\nBuffer zones (5 km): {len(gdf_projected)} polygons")


# 3. Bounding box filter: cx selector for northeast quadrant
ne_gdf = gdf.cx[-87.72:-87.58, 41.83:41.95]
print(f"\nNodes in NE quadrant: {list(ne_gdf['node'])}")


# 4. Nearest join: find closest substation to each demand point (in km).
demand_data = {"demand_id": ["D1", "D2", "D3"], "load_mw": [45, 70, 30]}
demand_lons = [-87.66, -87.75, -87.61]
demand_lats = [ 41.87,  41.86,  41.81]
demand_gdf = frame_from_xy(demand_data, demand_lons, demand_lats, crs="EPSG:4326")

demand_m = demand_gdf.to_crs("EPSG:3857")
substations_m = gdf.where(type="sub").to_crs("EPSG:3857")
nearest_m = demand_m.nearest_join(substations_m, how="left", rsuffix="sub")
nearest = nearest_m.to_crs("EPSG:4326")
nearest["dist_km"] = [None if d is None else d / 1000.0 for d in nearest_m["distance_sub"]]

print("\nNearest substation assignments (km):")
print(frame_preview(nearest, ["demand_id", "node", "dist_km"]).to_string(index=False))


# 5. Spatial join: demand points within substation buffers
gdf_buf = gdf.select_columns(["node"]).to_crs("EPSG:3857").buffer(8000).to_crs("EPSG:4326")
sj = demand_gdf.spatial_join(gdf_buf.select_columns(["node"]), how="left", predicate="within", rsuffix="buf")

print("Demand within buffer zones:")
print(frame_preview(sj, ["demand_id", "node"]).to_string(index=False))


# 6. Dissolve by node type: aggregate capacity
dissolved = (
    pd.DataFrame(gdf.to_records())
    .groupby("type", as_index=True)
    .agg({"cap_mw": "sum", "cost": "mean"})
)
print("\nDissolved by type:")
print(dissolved[["cap_mw", "cost"]].to_string())


# 7. High-cost nodes filter
high_cost = gdf.where([value > 7.0 for value in gdf["cost"]])
print(f"\nHigh-cost nodes (cost > 7): {list(high_cost['node'])}")


# Save to file
gdf.to_file(str(OUTPUT_DIR / "d1-gp-nodes.geojson"), driver="GeoJSON")
print("\nWrote d1-gp-nodes.geojson")


# Inline visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

node_rows = gdf.to_records()
node_coords = [row["geometry"]["coordinates"] for row in node_rows]
node_x = [coord[0] for coord in node_coords]
node_y = [coord[1] for coord in node_coords]
node_costs = gdf["cost"]

scatter = axes[0].scatter(node_x, node_y, c=node_costs, cmap="RdYlGn_r", s=120)
fig.colorbar(scatter, ax=axes[0], label="Routing Cost")
for row in node_rows:
    x, y = row["geometry"]["coordinates"]
    axes[0].annotate(row["node"], (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)

demand_rows = demand_gdf.to_records()
demand_coords = [row["geometry"]["coordinates"] for row in demand_rows]
axes[0].scatter(
    [coord[0] for coord in demand_coords],
    [coord[1] for coord in demand_coords],
    color="blue",
    s=80,
    marker="D",
    label="Demand",
)
axes[0].set_title("Utility Nodes (diamonds=demand points)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

gdf_sorted = gdf.sort_values("cost")
axes[1].barh(gdf_sorted["node"], gdf_sorted["cost"], color="#2563eb")
axes[1].set_xlabel("Routing Cost")
axes[1].set_title("Node Costs")
axes[1].grid(True, axis="x", alpha=0.3)

plt.suptitle("D1 Utilities: GeoPrompt Spatial Analysis", fontweight="bold")
plt.tight_layout()
plt.show()

# Basemap snapshot (real tiled basemap, saved as HTML)
try:
    import folium

    label_candidates = ["node", "stand_id", "asset_id", "zone_id"]
    label_col = next((c for c in label_candidates if c in gdf.columns), gdf.columns[0])
    centroids = gdf.geom.centroid()
    center_lat = float(np.mean([pt[1] for pt in centroids]))
    center_lon = float(np.mean([pt[0] for pt in centroids]))
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="CartoDB positron")

    for row in gdf.to_records():
        lon, lat = row["geometry"]["coordinates"]
        folium.CircleMarker(
            location=[float(lat), float(lon)],
            radius=6,
            color="#1d4ed8",
            fill=True,
            fill_opacity=0.85,
            popup=f"{label_col}: {row[label_col]}",
        ).add_to(fmap)

    optional_marker_specs = [
        ("demand_gdf", "demand_id", "blue", "bolt", "fa"),
        ("stations_gdf", "station_id", "green", "tree", "fa"),
        ("incidents_gdf", "inc_id", "red", "warning-sign", "glyphicon"),
        ("adapt_gdf", "site_id", "purple", "leaf", "fa"),
    ]

    for frame_name, id_key, color, icon, prefix in optional_marker_specs:
        frame = globals().get(frame_name)
        if frame is None:
            continue
        for row in frame.to_records():
            lon, lat = row["geometry"]["coordinates"]
            folium.Marker(
                location=[float(lat), float(lon)],
                icon=folium.Icon(color=color, icon=icon, prefix=prefix),
                popup=f"{id_key}: {row.get(id_key, 'n/a')}"
            ).add_to(fmap)

    map_path = OUTPUT_DIR / "d1-gp-basemap.html"
    fmap.save(str(map_path))
    print(f"Wrote basemap snapshot: {map_path.name}")
    from IPython.display import display
    display(fmap)
except Exception as exc:
    print(f"Basemap snapshot skipped: {exc}")

GeoPromptFrame:
 node  cost  cap_mw
PLANT   0.0     500
 SUB1   3.0     120
 SUB2   8.0      90
 SUB3  11.0      80
 SUB4   7.0     110
 SUB5   4.0     100
 SUB6  13.0      70

Buffer zones (5 km): 7 polygons

Nodes in NE quadrant: ['PLANT', 'SUB1', 'SUB2', 'SUB4']

Nearest substation assignments (km):
demand_id node  dist_km
       D1 SUB1 3.658949
       D2 SUB2 5.590141
       D3 SUB3 5.361589
Demand within buffer zones:
demand_id node
       D1 SUB1
       D2 SUB2
       D2 SUB5
       D3 SUB3
       D3 SUB4

Dissolved by type:
       cap_mw      cost
type                   
plant     500  0.000000
sub       570  7.666667

High-cost nodes (cost > 7): ['SUB2', 'SUB3', 'SUB6']

Wrote d1-gp-nodes.geojson
Wrote basemap snapshot: d1-gp-basemap.html


C:\Users\Matt\AppData\Local\Temp\ipykernel_23384\888792863.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section C: Scenario Comparison


In [8]:
scenarios = {

    "baseline":  {"reliability": 0.88, "outage_hrs_yr": 12.0, "cost_musd": 0.0},

    "hardened":  {"reliability": 0.94, "outage_hrs_yr":  5.0, "cost_musd": 28.0},

    "redundant": {"reliability": 0.97, "outage_hrs_yr":  2.5, "cost_musd": 55.0},

}

scenario_records = []

for name, vals in scenarios.items():

    score = round(vals["reliability"] * 0.5 + (1.0 / max(vals["outage_hrs_yr"], 0.1)) * 3 * 0.3

                  + (1.0 / max(vals["cost_musd"] + 1, 1)) * 20 * 0.2, 4)

    scenario_records.append({"scenario": name, **vals, "score": score})

scenario_records.sort(key=lambda r: -float(r["score"]))



fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = [r["scenario"] for r in scenario_records]

colors = ["#27ae60", "#e67e22", "#c0392b"]

axes[0].barh(names, [r["reliability"] for r in scenario_records], color=colors)

axes[0].set_xlabel("Reliability"); axes[0].set_title("Network Reliability"); axes[0].grid(True, axis="x", alpha=0.3)

axes[1].barh(names, [r["outage_hrs_yr"] for r in scenario_records], color=colors)

axes[1].set_xlabel("Outage Hours/Year"); axes[1].set_title("Annual Outages"); axes[1].grid(True, axis="x", alpha=0.3)

axes[2].barh(names, [r["score"] for r in scenario_records], color=colors)

axes[2].set_xlabel("Composite Score"); axes[2].set_title("Scenario Score"); axes[2].grid(True, axis="x", alpha=0.3)

plt.suptitle("D1 Utilities (GeoPrompt): Scenario Comparison", fontweight="bold")

plt.tight_layout(); plt.show()



(OUTPUT_DIR / "d1-gp-complex.json").write_text(

    json.dumps({"scenario_ranking": scenario_records}, indent=2, default=str), encoding="utf-8"

)

print("Wrote d1-gp-complex.json")

for r in scenario_records:

    print(f"  {r['scenario']}: score={r['score']}")


Wrote d1-gp-complex.json
  baseline: score=4.515
  redundant: score=0.9164
  hardened: score=0.7879


C:\Users\Matt\AppData\Local\Temp\ipykernel_23384\778923007.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
